# 📊 Fitness Supplements Market Intelligence
## US Market Analysis — 2021 to 2026

**Data source:** Google Trends (weekly search interest, United States)  
**Keywords:** creatine, protein powder, pre workout, weight loss supplements, omega 3  
**Period:** February 2021 – February 2026  
**Author:** Rodrigo Maximiliano Portillo

---

### Objectives
1. Identify which supplement categories are growing structurally vs temporarily
2. Quantify growth rates and compound annual growth (CAGR)
3. Map seasonal demand patterns to optimize media spend timing
4. Measure volatility and demand stability per category
5. Deliver actionable investment recommendations

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Analysis package ───────────────────────────────────────────────────────
from analysis import load_trends_5y, load_trends_12m
from analysis import (
    yearly_averages, cagr_table, yoy_momentum,
    seasonality_table, volatility_table, quarterly_averages, linear_trend,
    KEYWORDS, LABELS,
)

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#1A1A2E',
    'axes.facecolor':   '#16213E',
    'axes.edgecolor':   '#0F3460',
    'axes.labelcolor':  '#F5F5F5',
    'text.color':       '#F5F5F5',
    'xtick.color':      '#94A3B8',
    'ytick.color':      '#94A3B8',
    'grid.color':       '#0F3460',
    'grid.linewidth':   0.8,
    'font.family':      'sans-serif',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

COLORS = {
    'creatine':      '#E94560',
    'protein_powder':'#4A90D9',
    'pre_workout':   '#F59E0B',
    'omega_3':       '#10B981',
    'weight_loss':   '#94A3B8',
}
label_to_color = {LABELS[k]: COLORS[k] for k in KEYWORDS}

print('✅ Libraries and analysis package loaded')

---
## 1. Data Loading & Cleaning

In [ ]:
# ── Load datasets via analysis package ───────────────────────────────────
df    = load_trends_5y('../data/multiTimeline-lastfiveyears.csv')
df12m = load_trends_12m('../data/multiTimelinelast12monts.csv')

print(f'5-year  → {df.Week.min().strftime("%B %Y")} → {df.Week.max().strftime("%B %Y")}  ({len(df)} weeks)')
print(f'12-month → {df12m.Week.min().strftime("%B %Y")} → {df12m.Week.max().strftime("%B %Y")}  ({len(df12m)} weeks)')
print(f'Nulls (5y): {df[KEYWORDS].isnull().sum().sum()}')
print()
print('5-year averages (index 0–100):')
print(df[KEYWORDS].mean().round(1).rename(LABELS).sort_values(ascending=False).to_string())

---
## 2. Five-Year Trend — Structural Growth

In [ ]:
yearly = yearly_averages(df)

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#1A1A2E')

for kw in KEYWORDS:
    lw = 3.5 if kw == 'creatine' else 2
    alpha = 1.0 if kw == 'creatine' else 0.75
    ax.plot(yearly.index, yearly[kw], color=COLORS[kw],
            linewidth=lw, alpha=alpha, label=LABELS[kw], marker='o', markersize=5)

last_val = yearly['creatine'].iloc[-1]
ax.annotate(f'  {last_val:.0f}', xy=(yearly.index[-1], last_val),
            color=COLORS['creatine'], fontsize=12, fontweight='bold', va='center')

ax.axvline(x=2021.67, color='#FFFFFF', linestyle='--', linewidth=1, alpha=0.3)
ax.text(2021.72, 55, 'Creatine\ncrosses Protein\nSep 2021', color='#94A3B8', fontsize=8)

ax.set_title('US Fitness Supplements — 5-Year Search Interest', fontsize=15,
             fontweight='bold', color='#FFFFFF', pad=15)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Google Trends Index (0–100)', fontsize=11)
ax.legend(loc='upper left', framealpha=0.2, fontsize=10)
ax.set_ylim(0, 105)
ax.yaxis.set_major_locator(mticker.MultipleLocator(20))
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('../figures/01_five_year_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/01_five_year_trend.png')

---
## 3. CAGR — Compound Annual Growth Rate

In [ ]:
# ── CAGR table via analysis package ──────────────────────────────────────
cagr_df = cagr_table(yearly, start_year=2021, end_year=2026)
print(cagr_df.to_string(index=False))

# ── Chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#1A1A2E')

label_to_color_cagr = {LABELS[k]: COLORS[k] for k in KEYWORDS}
bar_colors = [label_to_color_cagr[lbl] for lbl in cagr_df['Keyword']]
bars = ax.barh(cagr_df['Keyword'], cagr_df['CAGR %'], color=bar_colors, height=0.55, zorder=3)

for bar, val in zip(bars, cagr_df['CAGR %']):
    color = '#E94560' if val > 0 else '#94A3B8'
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}%/yr', va='center', ha='left', fontsize=11,
            fontweight='bold', color=color)

ax.axvline(x=0, color='#94A3B8', linewidth=0.8)
ax.set_title('CAGR by Category — 2021 to 2026', fontsize=14, fontweight='bold',
             color='#FFFFFF', pad=12)
ax.set_xlabel('Compound Annual Growth Rate (%)', fontsize=11)
ax.set_xlim(-10, 42)
ax.grid(axis='x', alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig('../figures/02_cagr.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. YoY Momentum — Last 12 Months vs Previous 12 Months

In [ ]:
# ── YoY from 5-year series (via package) ─────────────────────────────────
yoy = yoy_momentum(df)
print('YoY momentum (5-year series):')
print(yoy.to_string(index=False))

print()

# ── YoY from dedicated 12-month CSV (cross-validation) ───────────────────
yoy_12m = yoy_momentum(df12m)
print('YoY momentum (12-month CSV — cross-check):')
print(yoy_12m.to_string(index=False))

# ── Chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#1A1A2E')

x = np.arange(len(yoy))
w = 0.35
b1 = ax.bar(x - w/2, yoy['Prev 12mo'], width=w, color='#0F3460', label='Prev 12 months', zorder=3)
b2 = ax.bar(x + w/2, yoy['Last 12mo'], width=w,
            color=[label_to_color[lbl] for lbl in yoy['Keyword']], label='Last 12 months', zorder=3)

for bar, pct in zip(b2, yoy['YoY Change %']):
    sign = '+' if pct > 0 else ''
    color = '#10B981' if pct > 0 else '#94A3B8'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{sign}{pct:.1f}%', ha='center', fontsize=10, fontweight='bold', color=color)

ax.set_xticks(x)
ax.set_xticklabels(yoy['Keyword'], fontsize=10)
ax.set_title('YoY Momentum — Last 12 Months vs Previous 12 Months', fontsize=14,
             fontweight='bold', color='#FFFFFF', pad=12)
ax.set_ylabel('Average Search Index', fontsize=11)
ax.legend(framealpha=0.2)
ax.grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig('../figures/03_yoy_momentum.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Seasonality — Monthly Demand Patterns

In [ ]:
# ── Seasonality via package ───────────────────────────────────────────────
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
seasonality = seasonality_table(df)

fig, axes = plt.subplots(2, 1, figsize=(13, 10))
fig.patch.set_facecolor('#1A1A2E')

ax = axes[0]
cr = seasonality['creatine']
bar_colors_s = ['#E94560' if v == cr.max() else
                '#F59E0B' if v >= cr.quantile(0.75) else '#0F3460' for v in cr]
bars = ax.bar(month_order, cr, color=bar_colors_s, width=0.65, zorder=3)
for bar, val in zip(bars, cr):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(val)), ha='center', fontsize=10, color='#F5F5F5')
ax.set_title('Creatine — Monthly Seasonality (5-Year Average)', fontsize=13,
             fontweight='bold', color='#FFFFFF', pad=10)
ax.set_ylabel('Avg Search Index', fontsize=11)
ax.set_ylim(0, 115)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.annotate('Peak: New Year', xy=(0, cr['Jan']), xytext=(1.5, cr['Jan'] + 8),
            arrowprops=dict(arrowstyle='->', color='#94A3B8'), color='#94A3B8', fontsize=9)
ax.annotate('Summer peak', xy=(6, cr['Jul']), xytext=(7.2, cr['Jul'] + 8),
            arrowprops=dict(arrowstyle='->', color='#94A3B8'), color='#94A3B8', fontsize=9)

ax2 = axes[1]
heat_data = seasonality[KEYWORDS].T
heat_data.index = [LABELS[k] for k in KEYWORDS]
sns.heatmap(heat_data, ax=ax2, cmap='RdYlGn', annot=True, fmt='.0f',
            annot_kws={'size': 9}, linewidths=0.5, linecolor='#1A1A2E',
            cbar_kws={'label': 'Search Index'})
ax2.set_title('All Keywords — Seasonal Heatmap', fontsize=13, fontweight='bold',
              color='#FFFFFF', pad=10)
ax2.tick_params(colors='#F5F5F5')

plt.tight_layout(pad=3)
plt.savefig('../figures/04_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nPeak months per keyword:')
for kw in KEYWORDS:
    peak = seasonality[kw].idxmax()
    print(f'  {LABELS[kw]:20s} → {peak} ({seasonality[kw].max():.0f})')

---
## 6. Volatility Analysis — Demand Stability

In [ ]:
# ── Volatility via package ────────────────────────────────────────────────
vol = volatility_table(df)
print(vol.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#1A1A2E')

bar_c = [label_to_color[lbl] for lbl in vol['Keyword']]
ax1.barh(vol['Keyword'], vol['CV %'], color=bar_c, height=0.5, zorder=3)
for i, (_, row) in enumerate(vol.iterrows()):
    ax1.text(row['CV %'] + 0.5, i, f"{row['CV %']:.1f}%", va='center',
             fontsize=10, fontweight='bold', color='#F5F5F5')
ax1.set_title('Coefficient of Variation\n(lower = more stable demand)',
              fontsize=12, fontweight='bold', color='#FFFFFF')
ax1.set_xlabel('CV % (Std Dev / Mean)', fontsize=10)
ax1.grid(axis='x', alpha=0.3, zorder=0)
ax1.axvline(x=40, color='#F59E0B', linestyle='--', linewidth=1, alpha=0.6)
ax1.text(40.5, -0.5, 'High volatility\nthreshold', color='#F59E0B', fontsize=8)

data = [df[k] for k in KEYWORDS]
bp = ax2.boxplot(data, vert=True, patch_artist=True,
                 medianprops=dict(color='white', linewidth=2))
for patch, kw in zip(bp['boxes'], KEYWORDS):
    patch.set_facecolor(COLORS[kw])
    patch.set_alpha(0.75)
ax2.set_xticklabels([LABELS[k] for k in KEYWORDS], rotation=15, ha='right', fontsize=9)
ax2.set_title('Distribution of Weekly Search Interest\n(5 years)',
              fontsize=12, fontweight='bold', color='#FFFFFF')
ax2.set_ylabel('Search Index', fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/05_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Quarterly Analysis — Where to Concentrate Budget

In [ ]:
# ── Quarterly via package ─────────────────────────────────────────────────
quarterly = quarterly_averages(df)
print(quarterly.rename(columns=LABELS).to_string())

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#1A1A2E')

x = np.arange(4)
w = 0.15
offsets = np.linspace(-2*w, 2*w, 5)
for i, kw in enumerate(KEYWORDS):
    ax.bar(x + offsets[i], quarterly[kw], width=w*0.9,
           color=COLORS[kw], label=LABELS[kw], zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(quarterly.index, fontsize=11)
ax.set_title('Average Search Interest by Quarter — 5-Year Average', fontsize=13,
             fontweight='bold', color='#FFFFFF', pad=12)
ax.set_ylabel('Avg Search Index', fontsize=11)
ax.legend(fontsize=9, framealpha=0.2, loc='upper right')
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.axvspan(-0.5, 0.5, alpha=0.07, color='#E94560')
ax.text(0, ax.get_ylim()[1] * 0.95, 'Best spend\nwindow', ha='center',
        color='#E94560', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/06_quarterly.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Trend Regression — Is Creatine Still Accelerating?

In [ ]:
# ── Linear regression via package ────────────────────────────────────────
result = linear_trend(df['creatine'])

x_vals = np.arange(len(df))
trend_line = result.slope * x_vals + result.intercept

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#1A1A2E')

ax.plot(df['Week'], df['creatine'], color=COLORS['creatine'], alpha=0.6,
        linewidth=1.5, label='Creatine (weekly)')
ax.plot(df['Week'], trend_line, color='#FFFFFF', linewidth=2, linestyle='--',
        label=f'Linear trend (slope = +{result.slope:.2f}/week)')
rolling = df['creatine'].rolling(52, center=True).mean()
ax.plot(df['Week'], rolling, color='#F59E0B', linewidth=2.5, label='52-week rolling avg')

ax.set_title(
    f'Creatine — Weekly Trend + Regression  |  R² = {result.r_value**2:.3f}  |  p < 0.001',
    fontsize=13, fontweight='bold', color='#FFFFFF', pad=12)
ax.set_ylabel('Search Index', fontsize=11)
ax.legend(framealpha=0.2, fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/07_regression.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Slope:       +{result.slope:.3f} index points per week')
print(f'Annualized:  +{result.slope * 52:.1f} index points per year')
print(f'R²:          {result.r_value**2:.3f}')
print(f'p-value:     {result.p_value:.2e}')
print(f'Conclusion:  Growth is statistically significant and shows no sign of plateauing.')

---
## 9. Strategic Summary Dashboard

In [ ]:
yearly = yearly_averages(df)
y2021 = yearly.loc[2021]
y2026 = yearly.loc[2026]
cutoff = df['Week'].max()
last12 = df[df['Week'] > cutoff - pd.DateOffset(months=12)]
prev12 = df[(df['Week'] > cutoff - pd.DateOffset(months=24)) &
            (df['Week'] <= cutoff - pd.DateOffset(months=12))]
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
seasonality = seasonality_table(df)
result = linear_trend(df['creatine'])

def _cagr_val(start, end, years=5):
    return ((end / start) ** (1 / years) - 1) * 100

fig = plt.figure(figsize=(14, 8))
fig.patch.set_facecolor('#1A1A2E')
fig.suptitle('US Fitness Supplements — Strategic Intelligence Dashboard',
             fontsize=15, fontweight='bold', color='#FFFFFF', y=0.98)

ax1 = fig.add_subplot(2, 3, (1, 2))
ax1.set_facecolor('#16213E')
for kw in KEYWORDS:
    lw = 3 if kw == 'creatine' else 1.5
    ax1.plot(df['Week'], df[kw], color=COLORS[kw], linewidth=lw,
             alpha=1 if kw == 'creatine' else 0.6, label=LABELS[kw])
ax1.set_title('5-Year Weekly Trend', fontsize=11, fontweight='bold', color='#FFFFFF')
ax1.legend(fontsize=7, framealpha=0.15, loc='upper left')
ax1.grid(alpha=0.2)

ax2 = fig.add_subplot(2, 3, 3)
ax2.set_facecolor('#16213E')
cagr_vals = [_cagr_val(y2021[k], y2026[k]) for k in KEYWORDS]
sorted_pairs = sorted(zip(cagr_vals, KEYWORDS), reverse=True)
sv, sk = zip(*sorted_pairs)
ax2.barh([LABELS[k] for k in sk], sv, color=[COLORS[k] for k in sk], height=0.5, zorder=3)
ax2.set_title('CAGR 2021→2026', fontsize=11, fontweight='bold', color='#FFFFFF')
ax2.set_xlabel('% / year', fontsize=9)
ax2.grid(axis='x', alpha=0.2, zorder=0)
for i, (v, k) in enumerate(zip(sv, sk)):
    ax2.text(v + 0.3, i, f'{v:+.1f}%', va='center', fontsize=9,
             color=COLORS[k], fontweight='bold')

ax3 = fig.add_subplot(2, 3, 4)
ax3.set_facecolor('#16213E')
cr_monthly = seasonality['creatine']
ax3.bar(range(12), cr_monthly,
        color=['#E94560' if v == cr_monthly.max() else
               '#F59E0B' if v >= cr_monthly.quantile(0.7) else '#0F3460'
               for v in cr_monthly], zorder=3)
ax3.set_xticks(range(12))
ax3.set_xticklabels(month_order, fontsize=7, rotation=45)
ax3.set_title('Creatine Seasonality', fontsize=11, fontweight='bold', color='#FFFFFF')
ax3.grid(axis='y', alpha=0.2, zorder=0)

ax4 = fig.add_subplot(2, 3, 5)
ax4.set_facecolor('#16213E')
yoy_vals = [(last12[k].mean() - prev12[k].mean()) / prev12[k].mean() * 100 for k in KEYWORDS]
ax4.bar([LABELS[k] for k in KEYWORDS], yoy_vals,
        color=['#10B981' if v > 0 else '#94A3B8' for v in yoy_vals], width=0.5, zorder=3)
ax4.axhline(0, color='#94A3B8', linewidth=0.8)
ax4.set_title('YoY Change (Last 12 Months)', fontsize=11, fontweight='bold', color='#FFFFFF')
ax4.set_ylabel('%', fontsize=9)
ax4.set_xticklabels([LABELS[k] for k in KEYWORDS], rotation=20, ha='right', fontsize=8)
ax4.grid(axis='y', alpha=0.2, zorder=0)

ax5 = fig.add_subplot(2, 3, 6)
ax5.set_facecolor('#16213E')
ax5.axis('off')
scorecard = [
    ('Creatine',          '+302%', 'CAGR +32.6%/yr', '#E94560', '🔥 SCALE'),
    ('Protein Powder',    '+117%', 'CAGR +16.8%/yr', '#4A90D9', '✅ GROW'),
    ('Omega 3',           '+78%',  'CAGR +12.3%/yr', '#10B981', '✅ GROW'),
    ('Pre Workout',       '-23%',  'Structural decline','#F59E0B','⚠️ TACTICAL'),
    ('Weight Loss Supps', '+73%*', 'YoY -18.2%',     '#94A3B8', '⛔ AVOID'),
]
ax5.set_xlim(0, 1)
ax5.set_ylim(0, 1)
ax5.set_title('Investment Scorecard', fontsize=11, fontweight='bold', color='#FFFFFF')
for i, (name, growth, note, color, decision) in enumerate(scorecard):
    y = 0.88 - i * 0.18
    ax5.add_patch(plt.Rectangle((0, y - 0.06), 1, 0.16, color='#0F3460', zorder=1))
    ax5.add_patch(plt.Rectangle((0, y - 0.06), 0.03, 0.16, color=color, zorder=2))
    ax5.text(0.06, y + 0.03, name, fontsize=8, fontweight='bold', color='#FFFFFF', va='center')
    ax5.text(0.06, y - 0.02, note, fontsize=7, color='#94A3B8', va='center')
    ax5.text(0.97, y, decision, fontsize=8, fontweight='bold', color=color, va='center', ha='right')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('../figures/08_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → figures/08_dashboard.png')

---
## 10. Key Findings & Recommendations

In [ ]:
yearly = yearly_averages(df)
y2021 = yearly.loc[2021]
y2026 = yearly.loc[2026]
cutoff = df['Week'].max()
last12 = df[df['Week'] > cutoff - pd.DateOffset(months=12)]
prev12 = df[(df['Week'] > cutoff - pd.DateOffset(months=24)) &
            (df['Week'] <= cutoff - pd.DateOffset(months=12))]
result = linear_trend(df['creatine'])
seasonality = seasonality_table(df)

def _cagr_val(start, end, years=5):
    return ((end / start) ** (1 / years) - 1) * 100

cr_cagr = _cagr_val(y2021['creatine'], y2026['creatine'])
pp_cagr = _cagr_val(y2021['protein_powder'], y2026['protein_powder'])
cr_yoy  = (last12['creatine'].mean()     - prev12['creatine'].mean())     / prev12['creatine'].mean()     * 100
wl_yoy  = (last12['weight_loss'].mean()  - prev12['weight_loss'].mean())  / prev12['weight_loss'].mean()  * 100

print('=' * 65)
print('  US FITNESS SUPPLEMENTS — KEY FINDINGS')
print('  February 2021 → February 2026')
print('=' * 65)

findings = [
    ('FINDING 1', 'Creatine is a structural market shift, not a trend',
     f'+302% total growth | CAGR {cr_cagr:.1f}%/yr | R² = {result.r_value**2:.3f} (p < 0.001)\n'
     f'Surpassed Protein Powder in September 2021. No plateau visible.'),

    ('FINDING 2', 'Weight loss supplements are in structural decline',
     f'YoY change: {wl_yoy:.1f}% | Lowest search index across all 5 keywords.\n'
     f'Avoid scaling any campaigns using weight loss framing.'),

    ('FINDING 3', 'January is the critical media window',
     f'Peak index: {seasonality["creatine"]["Jan"]:.0f}/100 in January (5-year avg)\n'
     f'Secondary peak: July–August. Lowest: December.'),

    ('FINDING 4', 'Women + creatine is the biggest untapped segment',
     '+123% YoY searches for "creatine for women" on X (Grok, Feb 2026)\n'
     '80-90% of creatine conversation is still male.\n'
     'Only 1 brand (GNC via creator) advertising to women — 50 days active.\n'
     'Just Ingredients holds longest-running ad: 739 days, clean label hook.'),

    ('FINDING 5', 'Protein powder is stable, not exciting',
     f'CAGR {pp_cagr:.1f}%/yr — growing but predictably. Lower urgency than creatine.\n'
     f'Opportunity: bundle with creatine to increase AOV.'),
]

for tag, title, detail in findings:
    print(f'\n  [{tag}]')
    print(f'  {title}')
    for line in detail.split('\n'):
        print(f'    → {line}')

print()
print('=' * 65)
print('  INVESTMENT RECOMMENDATION')
print('=' * 65)
print('  Creatine (women segment) → SCALE  | 40% | Q1 + Q3 priority')
print('  Creatine (men / gym)     → GROW   | 20% | Proven demand')
print('  Protein Powder           → GROW   | 15% | Bundle with creatine')
print('  Hispanic market          → TEST   | 10% | NaturalSlim 364d proves it')
print('  Omega 3                  → GROW   |  5% | Low competition')
print('  Pre Workout              → HOLD   |  5% | Tactical only')
print('  Weight Loss              → AVOID  |  0% | -18.2% YoY')
print('=' * 65)